In [ ]:
!pip install -q transformers==4.44.2 accelerate>=1.0 sentence-transformers

In [ ]:
import pickle
import torch
import torch.nn.functional as F

PATH_SOTTOGRAFO = "/kaggle/input/datasets/angelopaldino/datasetcora/cora_sottografo.pkl"
with open(PATH_SOTTOGRAFO, "rb") as f:
    sub = pickle.load(f)

testi = sub["testi"]
etichette = sub["etichette"]
grafo = sub["grafo"]
print(f"Nodi: {sub['n_nodi']}, Archi: {grafo.number_of_edges()}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

NOME_MODELLO = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(NOME_MODELLO)
model = AutoModelForCausalLM.from_pretrained(
    NOME_MODELLO,
    torch_dtype=torch.float16,
    device_map="auto",
)

In [ ]:

class ModelloPerplexity:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.device = model.device

    @torch.no_grad()
    def logprob_testo(self, testo_nodo, contesto=""):
        if contesto:
            prompt = contesto + "\n\n" + testo_nodo
            n_ctx_tok = self.tokenizer(contesto + "\n\n", return_tensors="pt").input_ids.shape[1]
        else:
            prompt = testo_nodo
            n_ctx_tok = 1

        ids = self.tokenizer(prompt, return_tensors="pt").input_ids.to(self.device)
        logits = self.model(ids).logits

        logprobs = F.log_softmax(logits[0, :-1], dim=-1)
        target = ids[0, 1:]
        tok_logprob = logprobs[torch.arange(target.shape[0]), target]

        nodo_logprob = tok_logprob[n_ctx_tok - 1:]
        somma = nodo_logprob.sum().item()
        n_token = nodo_logprob.shape[0]
        return somma, n_token

    def perplexity_testo(self, testo_nodo, contesto=""):
        somma, n_token = self.logprob_testo(testo_nodo, contesto)
        if n_token == 0:
            return float("inf")
        return float(torch.exp(torch.tensor(-somma / n_token)))

In [ ]:
m = ModelloPerplexity(model, tokenizer)

contesto = "Reinforcement learning is a branch of machine learning where agents learn by trial and error."
testo_coerente = "The agent maximizes a reward signal through interaction with the environment."
testo_incoerente = "The recipe requires two cups of flour and a pinch of salt."

print("PPL coerente:  ", round(m.perplexity_testo(testo_coerente, contesto), 2))
print("PPL incoerente:", round(m.perplexity_testo(testo_incoerente, contesto), 2))

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

# embedding del testo di ogni nodo (troncato per uniformita')
testi_per_embedding = [t[:1000] if t else "" for t in testi]
embeddings = embedder.encode(
    testi_per_embedding,
    convert_to_numpy=True,
    show_progress_bar=True,
    batch_size=64,
)
print("Shape embeddings:", embeddings.shape)  # (279, 384)

In [ ]:
import numpy as np
from tqdm import tqdm


class ScorerPerplexity:
    def __init__(self, modello, testi, embeddings, max_nodi_contesto=25, max_char_testo=400):
        self.modello = modello
        self.testi = testi
        self.embeddings = embeddings
        self.max_nodi_contesto = max_nodi_contesto
        self.max_char_testo = max_char_testo

    def _costruisci_contesto(self, nodo, membri_community):
        altri = [m for m in membri_community if m != nodo]
        if not altri:
            return ""
        # centroide semantico della community (esclusi il nodo stesso)
        centroide = self.embeddings[altri].mean(axis=0)
        # scelgo i nodi piu' vicini al centroide (cosine similarity)
        emb_altri = self.embeddings[altri]
        sim = emb_altri @ centroide / (
            np.linalg.norm(emb_altri, axis=1) * np.linalg.norm(centroide) + 1e-8
        )
        k = min(self.max_nodi_contesto, len(altri))
        idx_top = np.argsort(sim)[::-1][:k]
        selezionati = [altri[i] for i in idx_top]
        pezzi = [self.testi[m][:self.max_char_testo] for m in selezionati if self.testi[m]]
        return "\n\n".join(pezzi)

    def perplexity_nodo(self, nodo, membri_community):
        testo = self.testi[nodo][:self.max_char_testo]
        if not testo:
            return None
        contesto = self._costruisci_contesto(nodo, membri_community)
        return self.modello.perplexity_testo(testo, contesto)

    def ppl_partizione(self, partizione, mostra_avanzamento=True, descrizione="PPL(C)"):
        community = {}
        for nodo, c in enumerate(partizione):
            community.setdefault(c, []).append(nodo)

        ppl_per_nodo = {}
        iteratore = range(len(partizione))
        if mostra_avanzamento:
            iteratore = tqdm(iteratore, desc=descrizione, unit="nodo")

        for nodo in iteratore:
            c = partizione[nodo]
            ppl = self.perplexity_nodo(nodo, community[c])
            if ppl is not None:
                ppl_per_nodo[nodo] = ppl

        if not ppl_per_nodo:
            return float("inf"), {}
        media = float(np.mean(list(ppl_per_nodo.values())))
        return media, ppl_per_nodo

In [ ]:
#scorer = ScorerPerplexity(m, testi, embeddings)

#print("Calcolo PPL(C) con ground-truth...")
#ppl_gt, _ = scorer.ppl_partizione(etichette, descrizione="PPL ground-truth")

#print("Calcolo PPL(C) con partizione casuale...")
#rng = np.random.default_rng(0)
#part_random = rng.integers(0, len(set(etichette)), size=len(etichette))
#ppl_rand, _ = scorer.ppl_partizione(part_random, descrizione="PPL casuale")

#print(f"\nPPL(C) ground-truth: {ppl_gt:.3f}")
#print(f"PPL(C) casuale:      {ppl_rand:.3f}")
#print(f"Divario: {ppl_rand - ppl_gt:.3f} ({100*(ppl_rand-ppl_gt)/ppl_gt:.1f}%)")

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx

BASE = "/kaggle/input/datasets/angelopaldino/datasetcora"

# ordine nodi da cora.content
content = pd.read_csv(f"{BASE}/cora.content", sep="\t", header=None)
paper_ids = content.iloc[:, 0].astype(str).values
n_nodi = len(paper_ids)
id_to_index = {pid: i for i, pid in enumerate(paper_ids)}

# archi da cora.cites
cites = pd.read_csv(f"{BASE}/cora.cites", sep="\t", header=None, names=["cited", "citing"], dtype=str)
G = nx.Graph()
G.add_nodes_from(range(n_nodi))
for cited, citing in zip(cites["cited"], cites["citing"]):
    if cited in id_to_index and citing in id_to_index:
        G.add_edge(id_to_index[citing], id_to_index[cited])

# testo + etichette dal CSV combinato, ordinato per id
tape = pd.read_csv(f"{BASE}/combined.csv").sort_values("id").reset_index(drop=True)

testi = []
for i in range(n_nodi):
    t = tape.loc[i, "T"]; a = tape.loc[i, "A"]
    t = "" if pd.isna(t) else str(t).strip()
    a = "" if pd.isna(a) else str(a).strip()
    if t and a: testi.append(f"{t}. {a}")
    elif t: testi.append(t)
    elif a: testi.append(a)
    else: testi.append("")

etichette = tape["label"].values.astype(int)
print(f"Nodi: {n_nodi}, Archi: {G.number_of_edges()}, Classi: {len(set(etichette))}")

In [ ]:
testi_per_embedding = [t[:1000] if t else "" for t in testi]
embeddings = embedder.encode(testi_per_embedding, convert_to_numpy=True, show_progress_bar=True, batch_size=64)
print("Shape embeddings:", embeddings.shape)  # (2708, 384)

In [ ]:
scorer = ScorerPerplexity(m, testi, embeddings)


In [ ]:
print("PPL(C) ground-truth sul grafo completo...")
ppl_gt, ppl_nodi_gt = scorer.ppl_partizione(etichette, descrizione="PPL ground-truth")

print("PPL(C) casuale sul grafo completo...")
rng = np.random.default_rng(0)
part_random = rng.integers(0, len(set(etichette)), size=len(etichette))
ppl_rand, _ = scorer.ppl_partizione(part_random, descrizione="PPL casuale")

print(f"\nPPL(C) ground-truth: {ppl_gt:.3f}")
print(f"PPL(C) casuale:      {ppl_rand:.3f}")
print(f"Divario: {ppl_rand - ppl_gt:.3f} ({100*(ppl_rand-ppl_gt)/ppl_gt:.1f}%)")

In [ ]:
import json
risultati = {
    "ppl_ground_truth": ppl_gt,
    "ppl_casuale": ppl_rand,
    "divario_assoluto": ppl_rand - ppl_gt,
    "divario_percentuale": 100*(ppl_rand-ppl_gt)/ppl_gt,
    "n_nodi": n_nodi,
}
with open("/kaggle/working/risultati_grafo_completo.json", "w") as f:
    json.dump(risultati, f, indent=2)

# salvo anche le perplessita' per nodo (utili per la relazione)
np.save("/kaggle/working/ppl_per_nodo_gt.npy", np.array([ppl_nodi_gt.get(i, np.nan) for i in range(n_nodi)]))
print("Risultati salvati in /kaggle/working/")

In [ ]:
import numpy as np
from tqdm import tqdm


class LocalMovingPerplexity:
    def __init__(self, scorer, grafo, partizione_iniziale, max_iter=10, min_delta=1e-3,
                 dir_salvataggio="/kaggle/working"):
        self.scorer = scorer
        self.grafo = grafo
        self.partizione = np.array(partizione_iniziale).copy()
        self.max_iter = max_iter
        self.min_delta = min_delta
        self.dir_salvataggio = dir_salvataggio
        self._cache = {}
        # tengo traccia della partizione migliore vista
        self.migliore_partizione = self.partizione.copy()
        self.migliore_ppl = float("inf")

    def _membri(self, community_id, partizione=None):
        p = self.partizione if partizione is None else partizione
        return [n for n in range(len(p)) if p[n] == community_id]

    def _ppl_nodo_in_community(self, nodo, community_id, membri_override=None):
        chiave = (nodo, community_id)
        if chiave in self._cache:
            return self._cache[chiave]
        membri = membri_override if membri_override is not None else self._membri(community_id)
        ppl = self.scorer.perplexity_nodo(nodo, membri)
        if ppl is None:
            ppl = float("inf")
        self._cache[chiave] = ppl
        return ppl

    def _community_vicine(self, nodo):
        vicine = {self.partizione[v] for v in self.grafo.neighbors(nodo)}
        vicine.add(self.partizione[nodo])
        return vicine

    def _invalida_cache_community(self, community_id):
        chiavi = [k for k in self._cache if k[1] == community_id]
        for k in chiavi:
            del self._cache[k]

    def ottimizza(self):
        storia = []
        for it in range(self.max_iter):
            spostamenti = 0
            ordine = list(range(len(self.partizione)))

            for nodo in tqdm(ordine, desc=f"Iter {it+1}/{self.max_iter}", unit="nodo"):
                com_attuale = self.partizione[nodo]
                candidate = self._community_vicine(nodo)
                if len(candidate) <= 1:
                    continue

                ppl_per_com = {}
                for c in candidate:
                    membri = self._membri(c)
                    if c != com_attuale:
                        membri = membri + [nodo]
                    ppl_per_com[c] = self._ppl_nodo_in_community(nodo, c, membri_override=membri)

                com_migliore = min(ppl_per_com, key=ppl_per_com.get)
                if com_migliore != com_attuale and \
                   ppl_per_com[com_attuale] - ppl_per_com[com_migliore] > self.min_delta:
                    self.partizione[nodo] = com_migliore
                    self._invalida_cache_community(com_attuale)
                    self._invalida_cache_community(com_migliore)
                    spostamenti += 1

            ppl_media, _ = self.scorer.ppl_partizione(
                self.partizione, mostra_avanzamento=True,
                descrizione=f"Calcolo PPL(C) fine iter {it+1}"
            )
            storia.append({"iterazione": it + 1, "spostamenti": spostamenti, "ppl_media": ppl_media})
            print(f"Iter {it+1}: spostamenti={spostamenti}, PPL(C)={ppl_media:.3f}")

            # aggiorno la partizione migliore se questa e' la piu' bassa finora
            if ppl_media < self.migliore_ppl:
                self.migliore_ppl = ppl_media
                self.migliore_partizione = self.partizione.copy()

            # salvo dopo ogni iterazione (protezione da interruzioni)
            np.save(f"{self.dir_salvataggio}/partizione_iter_{it+1}.npy", self.partizione)
            np.save(f"{self.dir_salvataggio}/partizione_migliore.npy", self.migliore_partizione)

            if spostamenti == 0:
                print("Nessuno spostamento: convergenza raggiunta.")
                break

        print(f"\nMigliore PPL(C) = {self.migliore_ppl:.3f}")
        # restituisco la partizione MIGLIORE, non l'ultima
        return self.migliore_partizione, storia

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx

BASE = "/kaggle/input/datasets/angelopaldino/datasetcora"

# ordine nodi da cora.content
content = pd.read_csv(f"{BASE}/cora.content", sep="\t", header=None)
paper_ids = content.iloc[:, 0].astype(str).values
n_nodi = len(paper_ids)
id_to_index = {pid: i for i, pid in enumerate(paper_ids)}

# archi da cora.cites
cites = pd.read_csv(f"{BASE}/cora.cites", sep="\t", header=None, names=["cited", "citing"], dtype=str)
grafo = nx.Graph()
grafo.add_nodes_from(range(n_nodi))
for cited, citing in zip(cites["cited"], cites["citing"]):
    if cited in id_to_index and citing in id_to_index:
        grafo.add_edge(id_to_index[citing], id_to_index[cited])

# testo + etichette da combined.csv, ordinato per id
tape = pd.read_csv(f"{BASE}/combined.csv").sort_values("id").reset_index(drop=True)
testi = []
for i in range(n_nodi):
    t = tape.loc[i, "T"]; a = tape.loc[i, "A"]
    t = "" if pd.isna(t) else str(t).strip()
    a = "" if pd.isna(a) else str(a).strip()
    if t and a: testi.append(f"{t}. {a}")
    elif t: testi.append(t)
    elif a: testi.append(a)
    else: testi.append("")
etichette = tape["label"].values.astype(int)

print(f"Nodi: {grafo.number_of_nodes()}, Archi: {grafo.number_of_edges()}")

In [ ]:
# parto dalla partizione di Louvain
part_iniziale = np.load("/kaggle/input/datasets/angelopaldino/baselinelouvain/baseline_louvain_C0.npy")

print("Lunghezza partizione:", len(part_iniziale))
print("Numero nodi del grafo:", grafo.number_of_nodes())
assert len(part_iniziale) == grafo.number_of_nodes(), "DISALLINEAMENTO: la partizione non corrisponde al grafo!"

# local-moving guidato dalla perplexity
lm = LocalMovingPerplexity(scorer, grafo, part_iniziale, max_iter=10)
partizione_finale, storia = lm.ottimizza()

import json
np.save("/kaggle/working/partizione_perplexity_finale.npy", partizione_finale)
with open("/kaggle/working/storia_local_moving.json", "w") as f:
    json.dump(storia, f, indent=2)
print("Risultati salvati in /kaggle/working/")

In [ ]:
import networkx as nx
 
def esporta_per_gephi(grafo, etichette, part_louvain, part_perplexity, output="/kaggle/working/cora_visualizzazione.gexf"):
    G = grafo.copy()
    for n in G.nodes():
        G.nodes[n]["categoria_vera"] = int(etichette[n])
        G.nodes[n]["community_louvain"] = int(part_louvain[n])
        G.nodes[n]["community_perplexity"] = int(part_perplexity[n])
    nx.write_gexf(G, output)
    print(f"Grafo esportato: {output}")
    print("  - categoria_vera")
    print("  - community_louvain")
    print("  - community_perplexity")

In [ ]:
esporta_per_gephi(grafo, etichette, part_iniziale, partizione_finale)

In [ ]:
# esporta gli embedding per il TensorFlow Embedding Projector (projector.tensorflow.org)
# produce due file:
#   vectors.tsv   -> i vettori embedding, uno per riga, valori separati da tab
#   metadata.tsv  -> le etichette per colorare i punti (categoria, community, titolo)

import numpy as np

NOMI_CLASSI = {
    0: "Case_Based", 1: "Genetic_Algorithms", 2: "Neural_Networks",
    3: "Probabilistic_Methods", 4: "Reinforcement_Learning",
    5: "Rule_Learning", 6: "Theory",
}

def esporta_per_projector(embeddings, etichette, testi,
                          part_louvain=None, part_perplexity=None,
                          out_dir="/kaggle/working"):
    # 1) file dei vettori
    with open(f"{out_dir}/vectors.tsv", "w") as f:
        for vec in embeddings:
            f.write("\t".join(str(x) for x in vec) + "\n")

    # 2) file dei metadati (prima riga = intestazioni delle colonne)
    with open(f"{out_dir}/metadata.tsv", "w", encoding="utf-8") as f:
        colonne = ["categoria", "titolo_breve"]
        if part_louvain is not None:
            colonne.append("louvain")
        if part_perplexity is not None:
            colonne.append("perplexity")
        f.write("\t".join(colonne) + "\n")

        for i in range(len(embeddings)):
            cat = NOMI_CLASSI.get(int(etichette[i]), str(etichette[i]))
            titolo = testi[i][:40].replace("\t", " ").replace("\n", " ") if testi[i] else "(vuoto)"
            riga = [cat, titolo]
            if part_louvain is not None:
                riga.append(str(int(part_louvain[i])))
            if part_perplexity is not None:
                riga.append(str(int(part_perplexity[i])))
            f.write("\t".join(riga) + "\n")

    print(f"Creati {out_dir}/vectors.tsv e {out_dir}/metadata.tsv")
    print("Caricali su https://projector.tensorflow.org (pulsante 'Load')")



In [ ]:
partizione_finale = np.load("/kaggle/input/datasets/angelopaldino/partizione/partizione_perplexity_finale.npy")
esporta_per_projector(embeddings, etichette, testi, part_iniziale, partizione_finale)